## TIFF plot

In [ ]:
# Settings + Data fetching
system("conda install -y conda-forge::r-rcpp conda-forge::openssl conda-forge::r-sf conda-forge::r-terra conda-forge::r-ncdf4")
system("conda install -y conda-forge::r-r.utils conda-forge::r-tidyverse conda-forge::libgdal-hdf5 conda-forge::r-ggplot2")
system("conda install -y conda-forge::r-lubridate conda-forge::r-rcolorbrewer conda-forge::r-lattice conda-forge::r-png")

In [ ]:
library(ncdf4)
library(R.utils)
library(tidyverse)
library(terra)     
library(dplyr)
library(sf)        
library(jsonlite) 
library(utils)
library(ggplot2)
# ---
library(lubridate) # lubridate: operate on date and times data
library(RColorBrewer) # RColorBrewer: create colour palettes for thematic maps
library(lattice) # lattice : visualization system for typical graphics

In [ ]:
# SET DIRECTORIES
workdir <- getwd()
dataDir <- paste(workdir,"Data",sep = "/")
outputDir <- paste(workdir,"outputs/collection",sep = "/")
scriptDir <- paste(workdir,"scripts",sep = "/")

In [ ]:
# GENERAL FUNCTIONS

# ==============================================================================
# CREATE REPERTORY
# ==============================================================================
create.directory <- function(name.directory){
    ifelse(!dir.exists(file.path(name.directory)),
        dir.create(file.path(name.directory)),
        "Directory Exists")
}

# ==============================================================================
# DOWNLOAD FILENAME
# ==============================================================================
download.filename <- function(filename, url){
    options(timeout = 600)  # 10 minutes
    
    if(file.exists(filename)){
        cat(filename, "is (are) already in your repertory.")
    } else {
        download.file(url, filename, mode = "wb")
        print('File Downloaded')
    }
}

# ==============================================================================
# UNZIP FILENAME
# ==============================================================================
unzip.file <- function(filename){
    filename.length <- nchar(filename)

    start <- 1
    end <- filename.length-3
    unzip.filename <- substr(filename,start, end)

    
    if(!file.exists(unzip.filename)){        
        R.utils::gunzip(filename, overwrite=FALSE, remove=TRUE, BFR.SIZE=1e+07)
        cat(filename, "successfully unzipped!")
    }

    return(unzip.filename)
}

# ==============================================================================
# MOVE TO DATA DIRECTORY 
# ==============================================================================
move.file <- function(filename, new.path){
    new.filename <- paste(new.path, filename, sep = "/")
    file.rename(from=filename, to=new.filename)
    return(new.filename)
}

In [ ]:
# exists("download.file")
global_topo_tiff_gz <- "global_topo.tiff.gz"

# ==============================================================================
# DOWNLOAD FILENAME
bathy.url <- "https://topex.ucsd.edu/pub/global_topo_tiff/global.tiff.gz"
download.filename(global_topo_tiff_gz, bathy.url)

# ==============================================================================
# UNZIP FILENAME
global_topo_tiff <- unzip.file(global_topo_tiff_gz)


In [ ]:
# ==============================================================================
# FILE INFO
# ==============================================================================
print(file.info(global_topo_tiff_gz) )
print(file.info(global_topo_tiff) )

In [ ]:
# MOVE TO DATA DIRECTORY 
global_topo_tiff_gz <- move.file(global_topo_tiff_gz, dataDir)
global_topo_tiff <- move.file(global_topo_tiff, dataDir)

## NetCDF (Copernicus Data Files)

In [ ]:
# 2.2 DOWNLOAD AND LOAD BATHYMETRY ----
bathy_file <- "global_topo_1min_topo_19_1.nc"
bathy.url <- "https://topex.ucsd.edu/pub/global_topo_1min/topo_19.1.nc"
download.filename(bathy_file, bathy.url)

In [ ]:
# MOVE TO DATA DIRECTORY 
bathy_file <- move.file(bathy_file, dataDir)

## Data Exploration

File exploration : ornldaac [github repository](https://github.com/ornldaac/netCDF_data_in_R/blob/master/netCDF_in_r_ornldaac_tutorial.md)

In [ ]:
cat("Content of", getwd(), ":\n", list.files(), "\n")
cat(bathy_file,"exists:", file.exists(bathy_file),"\n")
# Check for the file
cat("Size:", file.info(bathy_file)$size)
 # If size is 0 or very small, the file is broken.

In [ ]:
# Open the NetCDF file

nc_file <- nc_open(bathy_file)
print(nc_file)

# Structure of the file

names(nc_file)
# Names of the variables
names(nc_file$var)
# Names of the dimensions
names(nc_file$dim)

In [ ]:
# List the attributes and sub-attributes

i <- 1
for(listVar in names(nc_file)){
    cat(i, listVar,"\n")
    for(listNames in names(nc_file[[listVar]])){
        cat("Attr:", listNames, ":", names(nc_file[[listVar]][[listNames]]),"\n")
    }
    i <- i + 1
}

In [ ]:
# Sub-Content for lat and lon
names(nc_file$dim$lon)
names(nc_file$dim$lat)

nc_file$dim$lon[1:5]
nc_file$dim$lat[1:5]

# 5 first Values
nc_file$dim$lon$vals[1:5] # print(nc_file$dim['lon'])
nc_file$dim$lat$vals[1:5]
nc_file$var$z$id[1:5]


In [ ]:
# Get coordinates variables
longitude <- ncvar_get(nc_file,"lon")
latitude <- ncvar_get(nc_file,"lat")
z <- ncvar_get(nc_file,"z")

In [ ]:
# Dimensions of latitude & longitude
print(c(length(longitude), length(latitude)))
print(dim(z))

# Check longtitude and latitude values
cat(" Head of longitude:",head(longitude),"\n")
cat(" Head of latitude:",head(latitude),"\n")

fillvalue <- ncatt_get(nc_file, "z", "_FillValue") # The fill value (aka, the no data value) is -9999.
print(fillvalue$value)

In [ ]:
nc_close(nc_file)

In [ ]:
z[z == fillvalue$value] <- NA
nc.slice.min80 <- z[,1]
dim(nc.slice.min80)

### Plots

In [ ]:
Bathy <- rast(bathy_file)
print("Converted into Raster File")

In [ ]:
# ==============================================================================
# 3. MAP PLOTTING ----
# ==============================================================================
png <- 1
if (png) {
  png(filename = "outputs/collection/Fig1.png", width = 1080, height = 720)
    
} else {
  pdf("outputs/collection/Fig1.pdf")
}

Depth_cuts <- c(-8200 ,-7000 ,-6000 ,-5000, -4000, -3000, -1800, -1400, -1000,  -600,  -400 , -200  ,   0 ,   50  , 250   ,500)
Depth_cols <- c(
  "#D6EAF8", "#AED6F1", "#85C1E9", "#5DADE2", "#3498DB",
  "#5DADE2", "#85C1E9", "#A9CCE3", "#D4E6F1", "#EBF5FB",
  "#F4F6F7", "#F8F9F9", "#FDFEFE", "#F2F3F4", "#EAEDED"
)

# Plot bathymetry
plot(Bathy,breaks = Depth_cuts, col = Depth_cols, legend = FALSE, axes = FALSE, box = FALSE,mar=c(0,0,0,0))

# Save the plot ---
dev.off()


### Convert NetCDF to CSV
Based on [Copernicus Marine Services](https://help.marine.copernicus.eu/en/articles/6328012-how-to-convert-netcdf-to-csv-using-r)

In [ ]:
#  Extract the coordinates
nc_file <- nc_open(bathy_file)

dim_lon <- ncvar_get(nc_file, "lon", collapse_degen=FALSE)
dim_lat <- ncvar_get(nc_file, "lat", collapse_degen=FALSE)
dim_depth <- ncvar_get(nc_file, "z", collapse_degen=FALSE)

In [ ]:
dim_lon[1:5]
dim_lat[1:5]
dim_depth[1:5]

In [ ]:

check_dim <- function(dim_var){
    cat(str(dim_var),
    length(dim_var),
    any(is.na(dim_var)),sep="\n")
}

check_dim(dim_lon)
print("---*---")
check_dim(dim_lat)
print("---*---")
check_dim(dim_depth)

In [ ]:
dim(dim_depth)

In [ ]:
coords <- expand.grid(dim_lon, dim_lat)
depth_matrix <- data.frame(cbind(coords, as.vector(dim_depth)))

In [ ]:
names(depth_matrix) <- c("lon", "lat", "depth")
head(depth_matrix)
# head(df.depth)
nc_close(nc_file)

In [ ]:
create.directory(outputDir)

head(na.omit(depth_matrix), 5)  # Display some non-NaN values for a visual check
csv_fname <- paste(outputDir,"netcdf_depth.csv", sep="/")
write.table(depth_matrix, csv_fname, row.names=FALSE, sep=";")